# 도로 재비산먼지 측정 데이터 정제·집계 보고서

## 1. 데이터 개요
- 출처: 한국환경공단 도로 재비산먼지 측정 정보
- URL: https://www.data.go.kr/data/15021888/fileData.do
- 이 데이터를 선택한 이유:

평소 금융 데이터와 공공데이터(환경, 보건의료) 분야에 관심이 있어, 처음에는 신용카드 관련 금융 데이터(사기 탐지용)를 검토했습니다. 그러나 해당 데이터는 개인정보 보호를 위해 컬럼명이 V1~V28로 익명화되어 있고 대부분 수치형 변수로만 구성되어 있어, 이번 EDA 보고서에서 다루고자 하는 목적(범주형·수치형·날짜 컬럼을 고루 활용한 분석)과는 맞지 않다고 판단했습니다.

이에 두번째로 관심 있던 환경 데이터를 선택하게 되었으며, 본 전공인 보건환경융합과학부에서도 자주 다루었던 주제라 친숙하게 접근할 수 있었습니다. 무엇보다 범주형(지역, 오염범례 등), 수치형(기온, 습도, 농도 등), 날짜/시간 컬럼이 골고루 포함되어 있어 이번 실습 목적에 가장 적합하다고 판단해 최종적으로 선정하였습니다.

- 각 컬럼 설명:

| 컬럼 | 의미 |
|---|---|
| 측정일자 | 측정한 날짜 |
| 측정시간 | 측정한 시각 |
| 지역 | 측정한 광역시도 |
| 지역명 | 측정한 시군구 |
| 도로명 | 측정한 도로 이름 |
| 시작점 | 측정 구간의 시작 지점 |
| 종점 | 측정 구간의 끝 지점 |
| 기온 | 측정 당시 기온 |
| 습도 | 측정 당시 습도 |
| 재비산먼지 평균농도 | 도로 위 먼지가 차량 통행으로 재부유하며 측정된 평균 농도 |
| 오염범례 | 농도값을 기준으로 매긴 등급 (매우좋음/좋음/보통/나쁨/매우나쁨) |

- 행의 의미: 이 데이터의 한 행은 **특정 날짜·시각에, 특정 도로 구간에서 이루어진 재비산먼지 측정 1건**을 의미합니다. 즉, 측정차가 어떤 도로의 시작점부터 종점까지 지나가면서 그 구간의 먼지 농도를 잰 기록 하나하나가 한 행에 해당합니다.
- 기간: 2026-06-01 ~ 2026-06-25 (25일)
- 대상 지역: 14개 시도
- 원본 행 수 / 정제 후 행 수: 1,472행 → 1,470행

## 2. 발견한 품질 문제
- **결측**: 지역명 컬럼에 2개의 결측값 존재 (전체의 0.14%)
- **중복**: 중복 행 없음 (0건)
- **날짜/시간 형식**: 측정일자와 측정시간이 별도 문자열 컬럼으로 분리되어 있어, 시간 흐름 분석을 위해서는 datetime 형식으로 변환이 필요했음
- **이상치**:
  - 기온·습도: IQR 기준 이상치 비율 0.0%로 계산됨
  - 재비산먼지 평균농도: 최댓값이 541로 평균(약 14)과 표준편차(약 31) 대비 크게 튀는 값이 존재해, 통계적으로는 이상치에 해당하는 값들이 꽤 있는 것 같음(약 9%)

## 3. 처리 결정과 근거
1. 지역명 결측 2건 → 삭제함 (전체 대비 비율이 매우 낮아 분석 결과에 미치는 영향이 미미하다고 판단)
2. 측정일자 + 측정시간 → datetime 형식으로 변환하여 각각 date, time컬럼으로 저장함 (시간대별·요일별 분석을 위해 필요)
3. 기온·습도 이상치 → 별도 처리하지 않음 (따로 해당 컬럼을 뽑아서 확인하니 통계적으로 이상치가 아닌 것 같아 그대로 두기로 결정)
4. 재비산먼지 평균농도 이상치 → 삭제하지 않고 그대로 둠 (해당 지표는 도로 통행량 급증, 공사 등 실제 상황에 따라 순간적으로 크게 튈 수 있는 값이라, 통계적 이상치가 곧 오류값을 의미하지 않는다고 판단함. 오히려 원인 파악이 필요한 유의미한 값일 가능성이 있음)

## 4. 주요 EDA 결과 정리
- **시간대별 패턴**: 하루 전반적으로 낮은 농도(대략 5~20 수준)를 유지하다, 오후 1~2시경 하나의 뚜렷한 스파이크(130 이상)가 관측됨. 출퇴근 시간대가 특별히 높다는 패턴은 나타나지 않았고, 특정 순간에 국지적으로 튀는 형태에 가까움
- **지역별 비교**: 상위 몇 개 지역(예: 고양시 등)이 다른 지역보다 확연히 높은 평균 농도를 보이며, 나머지 대부분 지역은 낮은 값에 몰려 있는 롱테일 분포를 보임
- **기온·습도와의 관계**: 재비산먼지 평균농도와는 기온(0.038), 습도(0.013) 모두 상관관계가 거의 없는 것으로 나타남. 날씨 조건이 먼지 농도를 직접적으로 설명하지는 못하는 것으로 보임
- **오염범례 분포**: 전체의 95.8%가 '매우좋음' 등급에 해당하며, 나쁨·매우나쁨 등급은 합쳐서 1% 내외에 불과함. 대부분의 시간·지역에서는 양호한 수준이나, 극히 일부 구간에서만 국지적으로 심각한 수치가 발생하는 구조로 해석됨

## 5. 한계와 더 하면 좋을 점

### 한계
1. 측정 기간이 25일(2026년 6월)로 짧고 단일 계절에 한정되어 있어, 이 결과가 계절적으로 대표성을 가지는지 판단할 수 없다.
2. 시간대별 스파이크가 반복되는 패턴인지, 특정일에 발생한 일회성 이벤트인지 구분할 수 없다.
3. 기온·습도와 먼지농도의 상관관계가 낮게 나왔다고 해서 날씨가 실제로 영향이 없다고 단정하기는 이르다. 도로 통행량, 강수 여부, 도로 포장 상태 등 상관관계 분석에 포함되지 않은 다른 요인이 실제 원인일 가능성이 있다.

### 더 하면 좋을 점
1. 여러 계절(봄·여름·가을·겨울)의 데이터를 추가로 확보하여 계절성 여부를 재검증한다.
2. 도로 통행량, 강수량 등 재비산먼지에 직접 영향을 줄 수 있는 변수를 추가로 확보하여 원인 분석을 보완한다.